# Concatenating Historical and Current MUR SST Icechunk Stores

This notebook demonstrates how to open and concatenate two MUR SST virtual Icechunk stores:
- **Historical store (v1)**: Contains data from 2002-06-01 to 2024-06-01
- **Current store (v2-p2)**: Contains data from mid-2024 onwards

Both stores can be opened using the same `helpers.get_repo()` function, making it straightforward to work with them together.

⚠️ You will need a [NASA Earthdata login username and password](https://urs.earthdata.nasa.gov/) and to run this notebook in the AWS us-west-2 region.

## Step 0. Install required libraries

In [ ]:
%%capture
!pip install icechunk==0.2.15 xarray==2025.4.0 zarr==3.0.8 earthaccess==0.14.0

## Step 1. Import libraries

In [ ]:
import icechunk as ic
from icechunk.credentials import S3StaticCredentials
from datetime import datetime
import matplotlib.pyplot as plt
from urllib.parse import urlparse
import earthaccess
from dask.diagnostics import ProgressBar
import xarray as xr
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

import helpers

## Step 2. Authenticate with NASA Earthdata

Login to Earthdata to get credentials for accessing the protected S3 buckets.

In [ ]:
earthaccess.login()
ea_creds = earthaccess.get_s3_credentials(daac='PODAAC')

## Step 3. Open the Historical Store (v1)

This store contains data from 2002-06-01 to 2024-06-01.

In [ ]:
# Historical store parameters
historical_bucket = 'nasa-eodc-public'
historical_store_name = "MUR-JPL-L4-GLOB-v4.1-virtual-v1"

# Open historical store using helper function
historical_repo = helpers.get_repo(
    bucket_name=historical_bucket, 
    store_name=historical_store_name, 
    ea_creds=ea_creds
)
historical_session = historical_repo.readonly_session(branch="main")
ds_historical = xr.open_zarr(historical_session.store, consolidated=False, zarr_format=3)

print(f"Historical store time range: {ds_historical.time.min().values} to {ds_historical.time.max().values}")
print(f"Historical store shape: {ds_historical.dims}")
ds_historical

## Step 4. Open the Current Store (v2-p2)

This store contains data from mid-2024 onwards.

In [ ]:
# Current store parameters
current_bucket = 'nasa-eodc-public'
current_store_name = "MUR-JPL-L4-GLOB-v4.1-virtual-v2-p2"

# Open current store using the same helper function
current_repo = helpers.get_repo(
    bucket_name=current_bucket,
    store_name=current_store_name,
    ea_creds=ea_creds
)

current_session = current_repo.readonly_session('main')
ds_current = xr.open_zarr(current_session.store, zarr_format=3, consolidated=False)

print(f"Current store time range: {ds_current.time.min().values} to {ds_current.time.max().values}")
print(f"Current store shape: {ds_current.dims}")
ds_current

## Step 5. Concatenate Both Stores

Combine both datasets along the time dimension to create a complete timeseries.

## Step 6. Verify Concatenation

Check that the time coordinate is properly ordered and there are no gaps or overlaps.

In [ ]:
# Concatenate along time dimension
ds_combined = xr.concat([ds_historical, ds_current], dim='time')

print(f"Combined store time range: {ds_combined.time.min().values} to {ds_combined.time.max().values}")
print(f"Combined store shape: {ds_combined.dims}")
print(f"Total timesteps: {len(ds_combined.time)}")
ds_combined

## Step 7. Example Analysis: Plot Complete Timeseries

Calculate and plot the mean SST over a region for the entire combined timeseries.

In [ ]:
import pandas as pd
import numpy as np

# Check for time ordering
time_diff = np.diff(ds_combined.time.values)
is_sorted = np.all(time_diff >= np.timedelta64(0, 'D'))
print(f"Time dimension is sorted: {is_sorted}")

# Check for duplicates
unique_times = len(np.unique(ds_combined.time.values))
total_times = len(ds_combined.time.values)
print(f"Unique timesteps: {unique_times}")
print(f"Total timesteps: {total_times}")
print(f"Has duplicates: {unique_times != total_times}")

# Check for gaps (assuming daily data)
time_deltas = pd.Series(time_diff)
expected_delta = np.timedelta64(1, 'D')
gaps = time_deltas[time_deltas > expected_delta]
if len(gaps) > 0:
    print(f"\nFound {len(gaps)} gaps in the timeseries:")
    print(gaps.head())
else:
    print("\nNo gaps found in the timeseries!")

## Summary

This notebook demonstrated how to:
1. Authenticate with NASA Earthdata
2. Open the historical MUR SST store (v1) using the `helpers.get_repo()` function
3. Open the current MUR SST store (v2-p2) using the same `helpers.get_repo()` function
4. Concatenate both stores along the time dimension with `xr.concat()`
5. Verify the concatenation is correct (no gaps, duplicates, or ordering issues)
6. Perform analysis on the complete combined dataset

Both stores can be opened using the same method, making it easy to work with them together.

In [ ]:
# Select a region (e.g., Caribbean)
lat_range = (10, 30)
lon_range = (-90, -60)

# Subset and calculate spatial mean
da_subset = ds_combined['analysed_sst'].sel(
    lat=slice(*lat_range), 
    lon=slice(*lon_range)
)

# Calculate mean over space
print("Computing spatial mean over time (this may take a few minutes)...")
with ProgressBar():
    timeseries = da_subset.mean(['lat', 'lon']).compute()

# Plot
plt.figure(figsize=(15, 5))
timeseries.plot()
plt.title(f'Complete MUR SST Timeseries ({lat_range[0]}°-{lat_range[1]}°N, {lon_range[0]}°-{lon_range[1]}°E)')
plt.ylabel('Sea Surface Temperature (K)')
plt.xlabel('Time')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\nTimeseries statistics:")
print(f"Mean: {timeseries.mean().values:.2f} K")
print(f"Min: {timeseries.min().values:.2f} K")
print(f"Max: {timeseries.max().values:.2f} K")